In [0]:
import pandas as pd
from pyspark.sql.types import StructType, StructField, DoubleType, LongType, StringType
from pyspark.sql.functions import col, from_unixtime, to_timestamp

In [0]:
apiKey="XQf22C5alIB5vXvUqKsjb4f7BZIQIKaH"

In [0]:
pip install -U polygon-api-client

In [0]:
from polygon import RESTClient
client = RESTClient(api_key=apiKey)

In [0]:
ticker = "AAPL"

# List Aggregates (Bars)
aggs = []
for a in client.list_aggs(ticker=ticker, multiplier=1, timespan="month", from_="2025-09-19", to="2025-09-20"):
    aggs.append(a)

schema = StructType([
    StructField("open", DoubleType(), True),
    StructField("high", DoubleType(), True),
    StructField("low", DoubleType(), True),
    StructField("close", DoubleType(), True),
    StructField("volume", LongType(), True),
    StructField("vwap", DoubleType(), True),
    StructField("timestamp", LongType(), True),
    StructField("transactions", LongType(), True),
    StructField("otc", StringType(), True),
])
aggs_dicts = [a._asdict() if hasattr(a, "_asdict") else a.__dict__ for a in aggs]

# Create DataFrame
df_pd = pd.DataFrame(aggs_dicts)
# Must set the timezone so to_timestamp uses IST
spark.conf.set("spark.sql.session.timeZone", "Asia/Kolkata")

spark_df = (
    spark
    .createDataFrame(df_pd, schema=schema)
    .withColumn("TimestampIst", (col("timestamp") / 1000).cast("double"))
    .withColumn("TimestampIst", to_timestamp(from_unixtime(col("TimestampIst"))))
)
spark_df.display()
